# Week 9 — Neural Pricing and Hedging

**Focus:** When is a learned pricing surrogate accurate enough for pricing and hedging decisions?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week09/week9_demos.ipynb)

## What this notebook is about

A learned pricing model is a numerical surrogate, not a replacement for financial reasoning. This notebook asks when a fast approximation is accurate enough for the decision that will use it.

You will train a neural approximation to Black–Scholes values, test it near and beyond the training boundary, translate prices into a delta-hedging policy, reconcile hedging profit and loss under costs, and compare constant no-trade bands across cost and rebalancing assumptions.

By the end, you should be able to:

- separate reference-model error, numerical error, surrogate error, and policy error;
- explain why random average accuracy can hide boundary failure;
- distinguish a pricing output from the hedge or action it supports;
- reconcile hedging P&L and implementation costs; and
- treat a no-trade band as a selected policy rather than a universal constant.

Black–Scholes supplies known reference values here. That makes this a controlled numerical study, not evidence that the same approximation will transfer to every market.

### Course context

For a concise review and the optional textbook route, see [Week 9 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week09/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 0 — Separate sampling error from discretization error — <a href="#Demo-0-%E2%80%94-Separate-sampling-error-from-discretization-error">Jupyter</a> · <a href="#scrollTo=md-436bfb4f0732">Colab</a></li>
<li>Demo 1 — Neural Black-Scholes pricer — <a href="#Demo-1-%E2%80%94-Neural-Black-Scholes-pricer">Jupyter</a> · <a href="#scrollTo=md-d547494df1a1">Colab</a></li>
<li>Demo 2 — Off-grid failure: train inside, test on the boundary — <a href="#Demo-2-%E2%80%94-Off-grid-failure%3A-train-inside,-test-on-the-boundary">Jupyter</a> · <a href="#scrollTo=md-67c51ccf390c">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 3 — BS delta hedge: cost-blind P&amp;L under proportional costs — <a href="#Demo-3-%E2%80%94-BS-delta-hedge%3A-cost-blind-P%26L-under-proportional-costs">Jupyter</a> · <a href="#scrollTo=md-697da908a4f4">Colab</a></li>
<li>Demo 4 — A transparent no-trade band helps under costs — <a href="#Demo-4-%E2%80%94-A-transparent-no-trade-band-helps-under-costs">Jupyter</a> · <a href="#scrollTo=md-2154e472b7a3">Colab</a></li>
<li>Demo 5 — The constant-band cost-and-frequency sweep — <a href="#Demo-5-%E2%80%94-The-constant-band-cost-and-frequency-sweep">Jupyter</a> · <a href="#scrollTo=md-33dd32304d78">Colab</a></li>
</ul>

## Project research-record checkpoint

Keep the project evidence trail in the single project-root `RESEARCH_RECORD.md`. Use this week's pricing and hedging exercises to add only relevant fields:

1. **Reference:** economic model, numerical reference solver, calibration inputs, and independent check.
2. **Surrogate:** training domain, boundary tests, approximation error, and versioned artifact.
3. **Policy:** hedge or action rule, information available at each rebalance, eligible instruments, and frequency.
4. **Implementation:** premium convention, costs, constraints, turnover, tail loss, and reconciliation.
5. **Changed-world test:** a boundary, discretization, volatility, cost, or frequency change that challenges the result.

Keep model, numerical, surrogate, and calibration error on separate ledgers. Append a decision-log entry for any project choice changed after development evidence.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from time import perf_counter
from scipy.stats import norm
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)


def save_lecture_figure(filename):
    """Save the current notebook figure beside the Week 9 lecture figures."""
    if "__file__" in globals():
        week_root = Path(__file__).resolve().parents[1]
    else:
        # A Jupyter kernel has no __file__. Preflight executes from lab/;
        # an uploaded standalone notebook instead writes to its local figures/.
        working = Path.cwd().resolve()
        week_root = working.parent if working.name == "lab" else working
    output = week_root / "figures" / filename
    output.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output, dpi=180, bbox_inches="tight")


def finish_plot():
    """Show interactively; close quietly during the headless validation run."""
    if "agg" in plt.get_backend().lower():
        plt.close()
    else:
        plt.show()


def bs_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

### Demo 0 — Separate sampling error from discretization error

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# Euler-discretized risk-neutral GBM, compared with the analytic Black-Scholes price.
# Repeated simulations let us audit nominal interval coverage separately from point error.
S0_mc, T_mc, r_mc, sigma_mc = 100.0, 1.0, 0.02, 0.20
strikes_mc = np.array([90.0, 100.0, 110.0])
reference_mc = bs_call(S0_mc, strikes_mc, T_mc, r_mc, sigma_mc)


def euler_call_grid(n_paths, n_steps, seed):
    rng_local = np.random.default_rng(seed)
    dt = T_mc / n_steps
    spots = np.full(n_paths, S0_mc)
    for _ in range(n_steps):
        spots *= 1 + r_mc * dt + sigma_mc * np.sqrt(dt) * rng_local.standard_normal(n_paths)
    discounted = np.exp(-r_mc * T_mc) * np.maximum(
        spots[:, None] - strikes_mc[None, :], 0.0
    )
    estimate = discounted.mean(axis=0)
    standard_error = discounted.std(axis=0, ddof=1) / np.sqrt(n_paths)
    return estimate, standard_error


numerical_rows = []
for n_paths in [2000, 8000]:
    for n_steps in [4, 52]:
        started = perf_counter()
        estimates, standard_errors, covered = [], [], []
        for repetition in range(30):
            estimate, standard_error = euler_call_grid(
                n_paths, n_steps, seed=10_000 * n_paths + 100 * n_steps + repetition
            )
            estimates.append(estimate)
            standard_errors.append(standard_error)
            covered.append(
                (reference_mc >= estimate - 1.96 * standard_error)
                & (reference_mc <= estimate + 1.96 * standard_error)
            )
        estimates = np.asarray(estimates)
        numerical_rows.append({
            "paths": n_paths,
            "time_steps": n_steps,
            "mean_abs_error": np.abs(estimates - reference_mc).mean(),
            "interval_coverage": np.asarray(covered).mean(),
            "runtime_seconds": perf_counter() - started,
        })

print("Model: risk-neutral GBM; European calls; Euler time stepping; analytic BS reference")
print(pd.DataFrame(numerical_rows).round(4).to_string(index=False))
# Expected: more paths primarily tighten sampling error; refining the time grid
# primarily reduces discretization bias. Runtime belongs on a separate ledger.

### Mini-project: build the first two ledgers

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 0 — Separate sampling error from discretization error**.

A network trained on an unmeasured numerical routine can reproduce its bias more
quickly, so measure the evaluator first. Using `week9_demos.ipynb`:

1. choose two path counts and two time-step counts;
2. price the same option grid;
3. compare with the reference evaluator;
4. report absolute error, interval coverage, and runtime separately.

**Research record:** the two path counts and two time-step counts; absolute error, interval coverage, and runtime for each grid, reported separately; the reference evaluator and the assumptions stated above the table.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 1 — Neural Black-Scholes pricer

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# Sample inputs from a parameter box
N = 12000
S = rng.uniform(80, 120, N)
K = rng.uniform(80, 120, N)
T = rng.uniform(0.05, 1.0, N)
r = np.full(N, 0.02)
sigma = rng.uniform(0.10, 0.40, N)
P = bs_call(S, K, T, r, sigma)

X_bs = np.column_stack([K / S, T, sigma])
P_normalized = P / S

split = int(0.85 * N)
scaler_bs = StandardScaler().fit(X_bs[:split])
X_bs_s = scaler_bs.transform(X_bs)
nn = MLPRegressor(hidden_layer_sizes=(64, 64), activation="tanh",
                  max_iter=800, random_state=0, early_stopping=True,
                  validation_fraction=0.1)
nn.fit(X_bs_s[:split], P_normalized[:split])
pred = nn.predict(X_bs_s[split:]) * S[split:]

rel_err = np.abs(pred - P[split:]) / np.maximum(P[split:], 0.1)
normalized_rmse = np.sqrt(np.mean(((pred - P[split:]) / S[split:]) ** 2))
mean_bp = np.mean(np.abs(pred - P[split:]) / S[split:]) * 1e4
print(f"Neural BS pricer  OOS normalized RMSE: {normalized_rmse:.4f}")
print(f"Neural BS pricer  OOS mean error: {mean_bp:.1f} bps of spot")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(P[split:], pred, s=4, alpha=0.4)
lim = max(P.max(), pred.max())
ax.plot([0, lim], [0, lim], "k--", lw=0.7)
ax.set_xlabel("True BS price"); ax.set_ylabel("Network prediction")
ax.set_title("Neural Black-Scholes — points cling to the 45° line")
save_lecture_figure("demo1_neural_pricer.png")
finish_plot()
# Expected: OOS normalized RMSE ~0.003 inside the training box -- strong interpolation,
#           which the next demo then breaks at the boundary.

### Demo 2 — Off-grid failure: train inside, test on the boundary

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# Train on part of sigma in [0.10, 0.30], preserve a fresh interior test there,
# and separately test sigma in (0.30, 0.40] as extrapolation.
mask_in = (sigma <= 0.30)
in_indices = np.flatnonzero(mask_in)
boundary_rng = np.random.default_rng(2026)
in_indices = boundary_rng.permutation(in_indices)
boundary_split = int(0.80 * len(in_indices))
boundary_train_idx = in_indices[:boundary_split]
boundary_test_idx = in_indices[boundary_split:]

scaler_boundary = StandardScaler().fit(X_bs[boundary_train_idx])
nn2 = MLPRegressor(hidden_layer_sizes=(64, 64), activation="tanh",
                   max_iter=800, random_state=0).fit(
                       scaler_boundary.transform(X_bs[boundary_train_idx]),
                       P_normalized[boundary_train_idx]
                   )

mask_out = (sigma > 0.30)
pred_out = nn2.predict(scaler_boundary.transform(X_bs[mask_out])) * S[mask_out]
worst_bp = np.max(np.abs(pred_out - P[mask_out]) / S[mask_out]) * 1e4
mean_bp = np.mean(np.abs(pred_out - P[mask_out]) / S[mask_out]) * 1e4
print(f"Off-grid mean error : {mean_bp:.1f} bps of spot")
print(f"Off-grid worst error: {worst_bp:.1f} bps of spot  "
      f"— the metric that matters for deployment")

# Compare fresh interior-test and off-grid error distributions in common units.
pred_in = nn2.predict(
    scaler_boundary.transform(X_bs[boundary_test_idx])
) * S[boundary_test_idx]
err_in_bp = (
    np.abs(pred_in - P[boundary_test_idx]) / S[boundary_test_idx] * 1e4
)
err_out_bp = np.abs(pred_out - P[mask_out]) / S[mask_out] * 1e4
print(f"Held-out interior mean error: {err_in_bp.mean():.1f} bps of spot")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(err_in_bp, bins=40, density=True, alpha=0.6, label="held-out interior")
ax.hist(err_out_bp, bins=40, density=True, alpha=0.6, label="off-box")
ax.set_xlabel("absolute error (bp of spot)"); ax.set_yscale("log")
ax.set_title("Off-grid errors live in a different distribution — Rule 26")
ax.legend(); save_lecture_figure("demo2_boundary_errors.png"); finish_plot()

### Mini-project: make the boundary visible

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demos 1–2 — Neural Black-Scholes pricer and off-grid failure**.

1. Fit the neural Black-Scholes surrogate in `week9_demos.ipynb`.
2. Hold out high-volatility or extreme-moneyness points by rule.
3. Plot interior and boundary error distributions.
4. Name the largest error in bp of spot.

**Research record:** the hold-out rule defining interior against boundary; interior and boundary error distributions; the largest error in basis points of spot; the deployment boundary it implies.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 3 — BS delta hedge: cost-blind P&L under proportional costs

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# A 1-year ATM call on GBM, weekly rebalance, varying transaction cost.
def bs_delta(S, K, T, sigma, r=0.0):
    if T <= 0:
        return (S > K).astype(float)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1)


def simulate_bs_hedge(n_paths, n_steps, S0=100.0, K=100.0, T=1.0,
                      sigma=0.20, cost_bps=50.0, bandwidth=0.0, seed=0,
                      premium=None, return_details=False, true_sigma=None,
                      hedge_sigma=None, jump_intensity=0.0, jump_sd=0.0):
    """Roll a (possibly bandwidth-regularized) delta hedge along GBM paths.

    bandwidth = 0  -> pure BS delta (rebalance every step)
    bandwidth > 0  -> hold inside the band; otherwise trade to its nearest edge
    Returns terminal P&L per path.
    """
    rng_local = np.random.default_rng(seed)
    dt = T / n_steps
    cost = cost_bps / 1e4
    true_sigma = sigma if true_sigma is None else true_sigma
    hedge_sigma = sigma if hedge_sigma is None else hedge_sigma
    if premium is None:
        premium = float(bs_call(S0, K, T, 0.0, sigma))
    S = np.full(n_paths, S0)
    cash = np.full(n_paths, premium)
    holdings = bs_delta(S, K, T, hedge_sigma)
    cash -= holdings * S  # initial buy
    cash -= np.abs(holdings) * S * cost  # initial trade cost
    turnover = np.abs(holdings)
    costs = np.abs(holdings) * S * cost
    for k in range(1, n_steps + 1):
        z = rng_local.standard_normal(n_paths)
        jump = np.where(
            rng_local.random(n_paths) < jump_intensity * dt,
            rng_local.normal(-0.5 * jump_sd ** 2, jump_sd, n_paths),
            0.0,
        )
        S = S * np.exp((-0.5 * true_sigma**2) * dt
                       + true_sigma * np.sqrt(dt) * z + jump)
        # Rebalance only before maturity. At maturity, liquidate the holding once.
        if k < n_steps:
            tau = T - k * dt
            target_delta = bs_delta(S, K, tau, hedge_sigma)
            if bandwidth > 0:
                lower = np.clip(target_delta - bandwidth, 0.0, 1.0)
                upper = np.clip(target_delta + bandwidth, 0.0, 1.0)
                target = np.clip(holdings, lower, upper)
            else:
                target = target_delta
            trade = target - holdings
            cash -= trade * S
            cash -= np.abs(trade) * S * cost
            turnover += np.abs(trade)
            costs += np.abs(trade) * S * cost
            holdings = target
    payoff = np.maximum(S - K, 0.0)
    # Liquidate at terminal
    cash += holdings * S
    cash -= np.abs(holdings) * S * cost
    turnover += np.abs(holdings)
    costs += np.abs(holdings) * S * cost
    pnl = cash - payoff
    if return_details:
        return pd.DataFrame({
            "pnl": pnl,
            "loss": -pnl,
            "turnover": turnover,
            "cost": costs,
        })
    return pnl


delta_detail = simulate_bs_hedge(
    5000, 52, cost_bps=50.0, bandwidth=0.0, seed=1,
    return_details=True,
)
pnl_delta = delta_detail["pnl"].to_numpy()
print(f"BS-delta hedge (50 bps, weekly): "
      f"mean PnL = {pnl_delta.mean():+.3f}, std = {pnl_delta.std():.3f}, "
      f"worst-5% mean loss = {-pnl_delta[pnl_delta <= np.quantile(pnl_delta, 0.05)].mean():.3f}, "
      f"mean turnover = {delta_detail['turnover'].mean():.3f}")

ledger_rows = []
for frequency in [12, 52, 252]:
    for fee in [0.0, 50.0]:
        detail = simulate_bs_hedge(
            3000, frequency, cost_bps=fee, bandwidth=0.0, seed=100 + frequency,
            return_details=True,
        )
        losses = detail["loss"].to_numpy()
        ledger_rows.append({
            "rebalances/yr": frequency,
            "cost_bps": fee,
            "mean_loss": losses.mean(),
            "loss_sd": losses.std(),
            "es5_mean_loss": losses[losses >= np.quantile(losses, 0.95)].mean(),
            "turnover": detail["turnover"].mean(),
        })
print("\nDelta hedge ledger (losses are positive; lower is better):")
print(pd.DataFrame(ledger_rows).round(3).to_string(index=False))
# Expected: costs turn a mean PnL near zero into a loss. More frequent rebalancing cuts
#           variance and raises cost.

### Mini-project: reconstruct the ledger

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 3 — Black-Scholes (BS) delta hedge under proportional costs**.

Using `week9_demos.ipynb`:

1. run delta hedging at monthly, weekly, and daily frequency;
2. repeat at zero and positive transaction cost;
3. report mean loss, dispersion, 5% expected shortfall, and turnover;
4. state all sign and unit conventions.

**Research record:** the three frequencies and two cost levels; mean loss, dispersion, 5% expected shortfall, and turnover for each; every sign and unit convention used.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 4 — A transparent no-trade band helps under costs

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# Select the bandwidth on validation paths, then test it on out-of-sample paths.
bands = np.linspace(0.0, 0.25, 51)
# Match the chapter and lecture's declared entropic-risk aversion. Keeping this
# single constant aligned matters: it changes both validation selection and the
# held-out cost-frequency comparison.
gamma = 0.5

def entropic_risk(losses, gamma=gamma):
    scaled = gamma * np.asarray(losses)
    anchor = scaled.max()
    return (anchor + np.log(np.mean(np.exp(scaled - anchor)))) / gamma

validation_rows = []
for b in bands:
    pnl = simulate_bs_hedge(4000, 52, cost_bps=50.0, bandwidth=b, seed=2)
    validation_rows.append({"bandwidth": b, "validation_risk": entropic_risk(-pnl)})
validation = pd.DataFrame(validation_rows)
selected_band = float(validation.loc[validation["validation_risk"].idxmin(), "bandwidth"])

rows = []
for name, band in [("delta", 0.0), ("selected band", selected_band)]:
    detail = simulate_bs_hedge(5000, 52, cost_bps=50.0, bandwidth=band, seed=3,
                               return_details=True)
    losses = detail["loss"].to_numpy()
    rows.append({
        "policy": name,
        "half_width": band,
        "entropic_risk": entropic_risk(losses),
        "mean_loss": losses.mean(),
        "es5_mean_loss": losses[losses >= np.quantile(losses, 0.95)].mean(),
        "turnover": detail["turnover"].mean(),
    })
df_band = pd.DataFrame(rows)
print(f"\nValidation-selected half-width: {selected_band:.3f}")
print(df_band.round(3).to_string(index=False))

holdings_grid = np.linspace(0, 1, 200)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for ax, spot in zip(axes, [85.0, 100.0, 115.0]):
    for tau in [0.1, 0.5, 1.0]:
        target_delta = float(bs_delta(np.array([spot]), 100.0, tau, 0.20)[0])
        lower = max(target_delta - selected_band, 0.0)
        upper = min(target_delta + selected_band, 1.0)
        trade = np.clip(holdings_grid, lower, upper) - holdings_grid
        ax.plot(holdings_grid, trade, label=f"T={tau:.1f}")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(f"spot = {spot:.0f}")
    ax.set_xlabel("current inventory")
axes[0].set_ylabel("trade")
axes[-1].legend()
fig.suptitle("The selected band creates an inspectable region of inaction")
fig.tight_layout(); save_lecture_figure("demo4_no_trade_band.png"); finish_plot()
# Expected: the validation-selected band lowers entropic risk and mean loss against the
#           delta hedge, and cuts turnover.

### Mini-project: compare mechanisms

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 4 — A transparent no-trade band helps under costs**.

1. Select a band on validation paths.
2. Freeze delta and band policies.
3. Plot trade against current inventory at three spot levels.
4. Compare entropic risk, mean loss, 5% expected shortfall, and turnover.

5. Identify where the policies disagree.

**Research record:** the band selected on validation paths; the frozen delta and band policies; trade against inventory at three spot levels; entropic risk, mean loss, 5% expected shortfall, and turnover.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 5 — The constant-band cost-and-frequency sweep

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
# 3x3 grid of (cost in bps) x (rebalance frequency per year).
# Select each bandwidth by validation entropic risk, then evaluate on fresh test paths.
costs_bps = [1, 25, 100]
freqs = [12, 52, 252]
band_grid = np.linspace(0.0, 0.25, 51)
best_bw = np.zeros((len(freqs), len(costs_bps)))
risk_improvement = np.zeros_like(best_bw)
for i, c in enumerate(costs_bps):
    for j, f in enumerate(freqs):
        validation_seed = 4600 + 100 * j + i
        best = (np.inf, 0.0)
        for bw in band_grid:
            pnl = simulate_bs_hedge(
                5000, f, cost_bps=c, bandwidth=bw, seed=validation_seed
            )
            risk = entropic_risk(-pnl)
            if risk < best[0]:
                best = (risk, bw)
        test_seed = 4700 + 100 * j + i
        pnl_delta = simulate_bs_hedge(
            16000, f, cost_bps=c, bandwidth=0.0, seed=test_seed
        )
        pnl_band = simulate_bs_hedge(
            16000, f, cost_bps=c, bandwidth=best[1], seed=test_seed
        )
        best_bw[j, i] = best[1]
        risk_improvement[j, i] = (
            entropic_risk(-pnl_delta) - entropic_risk(-pnl_band)
        )

frequency_labels = ["monthly", "weekly", "daily"]
cost_labels = [f"{c} bp" for c in costs_bps]
print("\nValidation-selected bandwidth:")
print(pd.DataFrame(best_bw, index=frequency_labels, columns=cost_labels).round(3))
print("\nTest entropic-risk improvement over delta — positive favors the band:")
print(pd.DataFrame(risk_improvement, index=frequency_labels,
                   columns=cost_labels).round(3))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharey=True)
images = [
    axes[0].imshow(best_bw, cmap="YlGn", aspect="auto", vmin=0),
    axes[1].imshow(risk_improvement, cmap="cividis", aspect="auto", vmin=0),
]
formats = [".3f", ".2f"]
titles = ["Validation-selected constant band", "Test risk improvement over delta"]
for ax, matrix, im, number_format, title in zip(
        axes, [best_bw, risk_improvement], images, formats, titles):
    ax.set_xticks(range(len(costs_bps)), cost_labels)
    ax.set_yticks(range(len(freqs)), frequency_labels)
    ax.set_xlabel("proportional cost")
    ax.set_title(title)
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            ax.text(col, row, format(matrix[row, col], number_format),
                    ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
axes[0].set_ylabel("rebalance frequency")
fig.tight_layout()
save_lecture_figure("demo5_cost_frequency_map.png")
finish_plot()
# Read: improvement is small at 1 bp and largest when trading is frequent and costly.

# Freeze the selected band in every cell and change the simulator. The hedge
# still uses 20% Black-Scholes delta; the world moves to 28% volatility with jumps.
stress_ratio = np.zeros_like(best_bw)
stress_band_loss = np.zeros_like(best_bw)
for i, c in enumerate(costs_bps):
    for j, f in enumerate(freqs):
        common = dict(
            n_paths=1500, n_steps=f, cost_bps=c, seed=900 + 10 * i + j,
            true_sigma=0.28, hedge_sigma=0.20, jump_intensity=0.5, jump_sd=0.12,
        )
        delta_loss = -simulate_bs_hedge(bandwidth=0.0, **common)
        band_loss = -simulate_bs_hedge(bandwidth=best_bw[j, i], **common)
        delta_tail = delta_loss[delta_loss >= np.quantile(delta_loss, 0.95)].mean()
        band_tail = band_loss[band_loss >= np.quantile(band_loss, 0.95)].mean()
        stress_ratio[j, i] = band_tail / delta_tail
        stress_band_loss[j, i] = band_tail

print("\nFrozen policies under higher volatility plus jumps: band/delta tail-loss ratio")
print(pd.DataFrame(stress_ratio, index=frequency_labels,
                   columns=cost_labels).round(2))
acceptance_line = 15.0
print(f"Absolute safety check: {np.sum(stress_band_loss <= acceptance_line)}/9 band cells "
      f"remain below a {acceptance_line:.1f} worst-tail-loss limit.")
# Expected: the band/delta loss ratio drops below 1 as costs rise and rebalancing
#           quickens. The band earns its place only under friction.

## Where this leaves us

A surrogate can be accurate on average and still fail where the decision is most sensitive. Pricing accuracy, hedge quality, turnover, and net cost are separate ledgers, and each must be tested on the domain in which the system will operate.

Week 10 moves from numerical inputs to financial documents, where representation and prediction must also respect publication and availability time.